# Public Portfolio Note

**Purpose:** Document the folium-based interactive map workflow for the Hospital Rating project.

**Inputs:** Excluded final analytic dataset with hospital coordinates, subgroup indicators, Google ratings, CMS ratings, and review counts.

**Outputs:** Local interactive HTML map for private review. Generated HTML files are excluded from the public repository.

**Public Data Limitation:** The map uses row-level hospital coordinates and API-derived rating fields, so the dataset and generated interactive HTML are not committed to the public repository.


# Section 7: Geospatial Visualization & System Distribution

This section documents the folium-based map workflow for exploratory local review. The public repository excludes the generated interactive HTML because it contains row-level hospital popups and API-derived fields.

**Key Steps:**
1. Load the excluded final analytic dataset.
2. Color markers by institutional subgroup.
3. Scale marker size by Google rating count.
4. Build local popups for private review.
5. Save the interactive HTML map outside the public data release.


In [ ]:
from pathlib import Path

import folium
from branca.element import MacroElement, Template
import numpy as np
import pandas as pd

# Public path constants. The row-level final analytic dataset is excluded.
PROJECT_ROOT = Path("..")
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MAP_DIR = OUTPUT_DIR / "maps"
FINAL_ANALYTIC_DATASET_PATH = "data/excluded/final_analytic_dataset.csv"
INTERACTIVE_MAP_OUTPUT_PATH = MAP_DIR / "interactive_hospital_map.html"

MAP_DIR.mkdir(parents=True, exist_ok=True)


def create_hospital_map(df, output_file=INTERACTIVE_MAP_OUTPUT_PATH):
    m = folium.Map(
        location=[36.7783, -119.4179],
        zoom_start=6,
        tiles="CartoDB positron",
    )

    color_map = {
        "Kaiser / integrated HMO": "#E8562A",
        "Academic": "#2C6FAC",
        "Other": "#AAAAAA",
    }
    scaling_factor = 0.18

    def get_star_html(rating):
        if pd.isna(rating):
            return "N/A"
        full_stars = int(rating)
        stars = "⭐" * full_stars + "☆" * (5 - full_stars)
        return f'<span style="color: #f39c12;">{stars}</span> {rating:.1f}'

    for _, row in df.iterrows():
        radius = np.sqrt(row["google_rating_count"]) * scaling_factor
        system_color = color_map.get(row["System"], "#888")

        popup_content = f"""
        <div style="font-family: sans-serif; width: 200px; line-height: 1.4;">
            <h4 style="margin: 0 0 8px 0; color: #2c3e50; border-bottom: 2px solid {system_color};">
                {row['cms_name']}
            </h4>
            <small><b>Google Star:</b> {get_star_html(row['google_star'])} ({int(row['google_rating_count'])} reviews)</small><br>
            <small><b>CMS Star:</b> {get_star_html(row['cms_star'])}</small><br>
            <small><b>Clinical Score:</b> {row['clinical_score']:.1f}/100</small><br>
            <small><b>Patient Exp:</b> {row['patient_exp_score']:.1f}/100</small>
        </div>
        """

        folium.CircleMarker(
            location=[row["latitude"], row["longitude"]],
            radius=max(radius, 3),
            popup=folium.Popup(popup_content, max_width=250),
            tooltip=row["cms_name"],
            color="black",
            weight=0.5,
            opacity=0.3,
            fill=True,
            fill_color=color_map.get(row["System"], "#AAAAAA"),
            fill_opacity=0.45,
        ).add_to(m)

    legend_html = """
    {% macro html(this, kwargs) %}
    <div style="
        position: fixed;
        bottom: 30px; left: 30px; width: 280px;
        background-color: white; border:1px solid #ccc; z-index:9999;
        padding: 10px; border-radius: 8px; font-family: sans-serif; font-size: 12px;
        box-shadow: 2px 2px 5px rgba(0,0,0,0.1);
        ">
        <b style="font-size:11px; display:block; margin-bottom:5px;">Hospital System</b>
        <div style="margin-bottom:3px;"><i style="background:#E8562A; width:10px; height:10px; border-radius:50%; display:inline-block; margin-right:5px;"></i> Kaiser / integrated HMO</div>
        <div style="margin-bottom:3px;"><i style="background:#2C6FAC; width:10px; height:10px; border-radius:50%; display:inline-block; margin-right:5px;"></i> Academic Medical Center</div>
        <div style="margin-bottom:8px;"><i style="background:#AAAAAA; width:10px; height:10px; border-radius:50%; display:inline-block; margin-right:5px;"></i> Other</div>

        <div style="border-top: 1px solid #eee; padding-top: 5px; color: #666; font-style: italic;">
            Note: Bubble size represents Google rating count.
        </div>
    </div>
    {% endmacro %}
    """

    macro = MacroElement()
    macro._template = Template(legend_html)
    m.get_root().add_child(macro)

    bounds = [
        df[["latitude", "longitude"]].min().values.tolist(),
        df[["latitude", "longitude"]].max().values.tolist(),
    ]
    m.fit_bounds(bounds)

    # Public copy: generated interactive HTML is for local/private review only and should not be committed.
    m.save(output_file)
    print(f"Public copy: local interactive map generated at {output_file}")
    return m


df = pd.read_csv(FINAL_ANALYTIC_DATASET_PATH)
create_hospital_map(df)
